In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
DATA_ROOT = "/Volumes/BlackPasspo/brisc/brisc"
from brisc.manuscript_analysis.utils import get_output_folder, get_path

save_path = get_output_folder(DATA_ROOT)
arial_font_path = None  # "/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf"


In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

from brisc.manuscript_analysis import cell_typing
from brisc.manuscript_analysis import spatial_plots_rabies as spatial

In [ ]:
adata = sc.read_h5ad(get_path('becalia_rabies_barseq/BRAC8498.3e/analysis/adata_q.h5ad', data_root=DATA_ROOT))

In [ ]:
atlas_size = 10

# Prepare bin_image for contour plotting
bin_image = spatial.prepare_area_labels(
    xpos=830,
    structures=[
        # Cortical plate layers (Allen CCF acronyms)
        "root",
        "CTX",
        "MB",
        "DG",
        "DG-mo",
        "DG-sg",
        "SCdg",
        "SCdw",
        "SCig",
        "SCiw",
        "SCop",
        "SCsg",
        "SCzo",
        "PAG",
        "MRN",
        "TH",
        "RN",
    ],
    atlas_size=10)

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

atlas_size = 10
chambers = ["chamber_07"]
clusters_not_used = ["Unassigned", "Zero_correlation", "VLMC"]
cortex_exclude = ["fiber_tract", "non_cortical", "TH", "hippocampal"]
qc = dict(best_score=0.3, knn_agree_conf=0.3, raw_gene_counts=2)

layer_tops = {
    "2/3": 116.8406715462,
    "4": 349.9050202564,
    "5": 477.8605504893,
    "6a": 717.1835081307,
    "6b": 909.8772394508,
    "wm": 957.0592130899,
}

cm = 1 / 2.54
fontsize_dict = {"title": 7, "label": 8, "tick": 6, "legend": 6}
plt.rcParams.update({
    "axes.titlesize": fontsize_dict["title"],
    "axes.labelsize": fontsize_dict["label"],
    "xtick.labelsize": fontsize_dict["tick"],
    "ytick.labelsize": fontsize_dict["tick"],
    "legend.fontsize": fontsize_dict["legend"],
    "legend.title_fontsize": fontsize_dict["legend"],
})

# ---- one mosaic: (coronal + KDE) per cluster ----
fig = plt.figure(figsize=(17.4 * cm, 12 * cm), dpi=300, constrained_layout=True)

# Choose how many (coronal+KDE) pairs per row:
# ncols=2 usually reads well because each cluster uses 2 columns internally.
gs = fig.add_gridspec(nrows=1, ncols=1)
pairs_per_row = 4

clusters, axd = cell_typing.plot_cluster_mosaic(
    fig,
    gs[0],
    adata,
    bin_image,
    fontsize_dict,
    group_key="custom_leiden",
    chambers=chambers,
    clusters_not_used=clusters_not_used,
    cortex_exclude=cortex_exclude,
    qc=qc,
    atlas_size=atlas_size,
    ncols=pairs_per_row,                 # <-- pairs per row
    layer_tops=layer_tops,
    x_min=1970,
    x_max=2260,
    bw_method=0.1,
    high_opacity_types=("Lamp5", "Pvalb", "Sst", "Vip", "L6b", "L5 NP"),
    s_default=0.8,
    alpha_default=0.05,
    s_high_opacity=0.5,     # smaller
    alpha_high_opacity=0.3, # more opaque
)

if axd:
    bottom_right_scatter_ax = list(axd.values())[-1][0]
    scalebar = AnchoredSizeBar(
        bottom_right_scatter_ax.transData,
        1000 / atlas_size,
        label=None,
        loc="upper left",
        pad=0.2,
        borderpad=0.3,
        color="black",
        frameon=False,
        size_vertical=4,
    )
    bottom_right_scatter_ax.add_artist(scalebar)

    for idx, (_, ax_kde) in enumerate(axd.values()):
        if idx % pairs_per_row == pairs_per_row - 1:
            ax_kde.set_ylim(1000, 0)
            ax_kde.set_yticks([0, 500, 1000])
            ax_kde.set_yticklabels(["0 mm", "0.5 mm", "1.0 mm"])
            ax_kde.tick_params(
                axis="y",
                which="both",
                left=False,
                labelleft=False,
                right=True,
                labelright=True,
                labelsize=fontsize_dict["tick"],
            )

plt.show()
fig.savefig(save_path / "suppfig7_custom_leiden_a4_mosaic_with_perpanel_kde.pdf", bbox_inches="tight")
